In [2]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


load_dotenv()

google_api_key = os.environ["GOOGLE_API_KEY"]

free_model_names = [
    # Current stable Gemini models
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",

    # Preview models
    "gemini-3-flash-preview",

    # Legacy models - restricted/deprecated
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
]

models = [
    ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=google_api_key,
        max_retries=0,
        timeout=30,
    )
    for model_name in free_model_names
]
llm = models[0].with_fallbacks(models[1:])

# Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.


In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
#message based summarization
agent = create_agent(
    llm,
    middleware = 
    [SummarizationMiddleware(
        model=llm,
        saver=InMemorySaver(),
        trigger=("messages",10),
        keep=('messages',4)
        )
    ]
)

In [6]:
# run with thread id 
config={"configurable":{"thread_id":"1"}}

In [7]:

# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='5aeb55a5-29b4-4677-ba6b-b8e92031d644'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'Es0CCsoCAWkUfRO4a5+vRMzzuWtN1vGOh+6JogRDntPv8Gdq6bIFBgYhj6z8AA2Qs3HAFpUK9HinZX8/RnmWd6hqLm/fzgxoSiB5+y9HKPHKeyyIxPzogEsFyKzJMY7OuGHQmCV8Mn2+8Lptgx78nBLqSgixM5L4b1eeIQRqlXAxD4WBdzUVHgZPSZgK/a2pnPaHGLt9oncGhVjuRjcj4Oy6hL3FSpV8BIDNUxf6r/3/p+JbEK0YQHXsyB/DKTKNyd5X9oK0B8gTAq977jTIq9yTfw32XiiC5geAMT1URgpHhXC63B688dVbjtfGY8hBSXiLpG+5mKqjy+NyjUwHfbRIkZZc0nGftcWvJ3FZjYxugeO62mPVukImmzcEqlOFSHuf3ElPb1Ypg7HzdqtlTKPkSKSHAGuYA4dWon9YU8iw9bT69dGVJRHnfD+dgOdT'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1ed9b-e242-7090-b3f3-cd17bf0f1f50-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 63, 'total_toke

# Restricting the context using token parameters

In [9]:

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~267 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='d3342a37-f68e-4638-aea7-f1b0e8e629ac'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_5696036291957508645': 'ErwCCrkCAWkUfRO8M3jPZUQlDPQDRdeOlrOrISzhD5veouMOstmMY6/UfVnzE1XwGfCqvaIItGK7ylv739iPkORDivPPCMw+0LNMAVWKr5scuI82LnJOKGZ26xqY7yjB3GYzHbficlTx4kFYh0O7PjFhb7tJOogP5Pgb/IEob5Oi50+FrwW2zP+36QC0Y9akgtRLCktrlT/CCeOy4FaxJ9PDjh/1q0JxsGCX9O9emgLAV4RUv7ePl2Y2zZyYWJypgXqlS7AqD7CKM1Vb5skPuuRZHjtQAe9Khl8Z4RX+QqrSDzgGp66MiBm4WAW0ugVA2+OEPx1NmP3pF+v+9LRK5flGvtu6rfH8Luwt+/56ETwPR/isEWaev+JMr305Z29D+0LY2ElY4BkCOVWC3qsN6xbK6YOuY0bGWhGXvwsa3A=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1edab-5394-7435-9f0b-f01b1443a646-0', tool_calls=[{'name

# Using fraction to restrict the context window

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~115 tokens (0.0898%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='53fdd794-7dea-4d63-932f-f0a8305d7453'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_9680058912478971999': 'EqYCCqMCAWkUfRO50gjF2eOH68HhmLmT90PAFMjcV9OS50oRwqm/m/tRsepMqY5D/PPlghfbWkHfOZNTXJUAytCbTY4Hjb2o6f0jz2fniTj97XvF9xqazuWVkoRISISYX/H4AoqT/upJabDdAq0USGCX/xVRKXowmdY+V7FOUX7ztcaPu9PNjgrFc3OZkna5VeKlb3X8gi66Vr2cf61PU7ae9ky6cltem6p8LS9OsAyAZArzwrEFyJ26UHxBOy7TNL/MDZZ/HtkhiTwbnAgkIfWRGzi5SaNEQLuhf3D+sM+58lcSLW8P5XkdQC4IGV1OhdTlhD/QHl1AnBuHLKTIaeguLdiN03cJu39vGucfNOfNpXJUS3SABQISYg3/mZZQlDQHalSrhUcD'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1edc4-f933-70c3-8379-641b12819536-0', tool_calls=[{'name': 'search_hotels', 'args': {'c

# Human in the loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

High-stakes operations requiring human approval (e.g. database writes, financial transactions).
Compliance workflows where human oversight is mandatory.
Long-running conversations where human feedback guides the agent.

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [15]:
agent=create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [16]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to mohitnaskar02@gmail.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [17]:
result

{'messages': [HumanMessage(content="Send email to mohitnaskar02@gmail.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='11f69364-641d-4f41-8891-90a2831cece8'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "mohitnaskar02@gmail.com"}'}, '__gemini_function_call_thought_signatures__': {'call_2097964800090359521': 'EokBCoYBAWkUfROjWv5fnF1aq1IFkiBSyp4NRS7MqcDTDDdFUfYllac7EJNY5jBdDL0fFyFNZTKKegEIjWhPaLLzh7kVedPcNkS6DF254eSwa4hIiX2E/lKafyrqn05JoFbZ6JpLQi3PMyMB4UJVSLx899CaEzDtZBVogk48HZMbnSvOvxNF6gVYntc='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1edcf-fc3c-77aa-af89-3283de20a2c4-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'mohitnaskar02@gmail.com'}, 'id'

In [18]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "The email has been successfully sent to mohitnaskar02@gmail.com with the subject 'Hello' and body 'How are you?'.", 'extras': {'signature': 'EokBCoYBAWkUfRMMCC06n2QqYzXbDWOxLHNcegPcoyU22jnZCvJDtqNy7ZE/ajKfqOI+U3CtwljexLFmoiEDgNU72Iek3F29CiMyJSfipeSGLTSanGAoZTCsteQBP40iInscjhlcgBYUHyV3jaCJNb23GZyKt6qiCrtbOqKKtEjezElTnHR0PPu4tzA='}}]


In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=llm,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [21]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [22]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email could not be sent because the tool call was rejected by the user. Let me know if you would like me to try again or perform a different task.', 'extras': {'signature': 'EokBCoYBAWkUfRNWACO3QyM2gZt0H4sS+4SpJVRA53yjE4Mgjkkj6EUgzAfasXENoNTLunpOCDcwv699iAgJamBW92KTkAqV1uJW4yyOG9GsLhUuYhm32rCZLs1RF7/Pdna8eBj+anKb0xj2kJx5GJjnC1guyPh+me4pb8nKKdAEwr3sKkE0W48m7oY='}}]


In [23]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c7c8affa-6dcb-4322-90f0-2d1a49cd0a40'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_6604077341510566317': 'EokBCoYBAWkUfROuAOuwDgCh5jn2g5/Epn2JGv75T7Df440A2nJhIJYKoCpr+qF7jY3jecDrgEtNg6EO6wtT9M2JFgS7FkDHFBpH6UDKUB4QT+Ew+TCXnYnqayOnHZvRiJRrSmOCiaOXxrwmfbX5cG3/Fmm02Wm+mfHDebFU2qw4OAWHSkDpPTp4Hzw='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1edd2-b1a2-74d1-a55e-d9ac501b63f0-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_6604077341510566317', 